# Feature Engineering: Gold Price Forecasting
Transform stationary return data into a forecasting-ready dataset by creating time-lagged features that enable prediction of future gold price movements without look-ahead bias.

---

## Core Strategy
1. Lag Feature Creation
    - What: Generate historical values (Lag 1–5 days) for each predictor (SPX_Return, USO_Return, SLV_Return, EURUSD_Return) and the target (GLD_Return).
    - Why: Financial markets exhibit temporal dependency. Lagged features allow the model to learn patterns like "If oil dropped 3% yesterday, how does gold typically react today?"
    - Output: 20 new lagged predictor columns + 5 autoregressive target lags.
2. Target Variable Definition
    - What: Shift the GLD_Return series by -1 to create a target column representing next-day return.
    - Why: Ensures the model learns to predict future values using past/present information only. This eliminates look-ahead bias, a critical error in time-series forecasting.
    - Output: A single target column aligned with lagged features.
3. Time-Based Train/Test Split
    - What: Split data chronologically (80% train, 20% test) without shuffling.
    - Why: Preserves temporal order and simulates real-world deployment where future data is unavailable during training. Prevents data leakage and ensures valid performance estimation.
    - Output: Two disjoint datasets with non-overlapping date ranges.
4. Artifact Persistence
    - What: Save X_train, y_train, X_test, y_test, and a feature_manifest.json to disk.
    - Why: Enables reproducibility across model notebooks. The manifest documents exactly which features were engineered, when, and with what parameters—critical for auditing and iteration.
    - Output: Serialized artifacts in `artifacts/feature-engineering/` ready for model training.

# **Import Libraries**

In [1]:
# data manipulation
import pandas as pd
import numpy as np

# directory handling
import sys
import os

# file handling
from pathlib import Path
import joblib
import json

# logging
from datetime import datetime

# **Configurations**

In [2]:
# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# directoties
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "feature-engineering"
DATA_DIR = PROJECT_ROOT / "data"

# Create directories if not exist
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# load gold price returns data
data_path = DATA_DIR / "gold_price_returns.csv"

df_returns = pd.read_csv(data_path, parse_dates=True, index_col=0)

In [5]:
# verify data loaded correctly
df_returns.head(10)

,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return
Date,,,,,
2008-01-03,0.000000,0.008367,-0.001274,0.006917,0.001902
2008-01-04,-0.024552,-0.005142,-0.013526,-0.007720,0.000679
2008-01-07,0.003223,-0.004229,-0.023412,-0.007516,-0.004875
2008-01-08,-0.018352,0.023711,0.007417,0.035674,0.060478
2008-01-09,0.013624,-0.002650,-0.010649,-0.004490,-0.058245
2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339
2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739
2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337
2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499


In [6]:
# lag features and target configuration
LAG_RANGE = range(1, 6)     # Lag 1-5 days
TARGET_SHIFT = -1           # Predict next day
TRAIN_RATIO = 0.8
TARGET_COL = "GLD_Return"
FEATURE_COLS = [col for col in df_returns.columns if col != 'GLD_Return']